# Cone-linearized center-disc analysis
**The experiment.** This notebook combines `LinearEquivalentDiscConeLin` with the newer
`LinearEquivalentDisc` implementation. Both show each natural-image center patch three ways:

| `stimulusTag` | what it is |
|---|---|
| `image` | the image patch itself |
| `intensity` | uniform center disc at the ordinary linear-equivalent intensity |
| `linConeIntensity` / `lin cone intensity` | uniform center disc at the **cone-linearized** equivalent intensity — averaged after a Weber cone nonlinearity `I / (I + WeberConstant)` |

The older, same-named `LinearEquivalentDisc` class only alternated `image` and `intensity`.
It did not define `linearizeCones`; discovery therefore excludes those blocks from every
section below using their saved block metadata rather than a hard-coded date cutoff.

The nonlinearity index is computed per `(imageName, patchIndex)`:

$$\mathrm{NLI} = \frac{\mathrm{image} - \mathrm{disc}}{|\mathrm{image}| + |\mathrm{disc}|}$$

and is set to zero when neither response clears the recording-mode threshold.


In [ ]:
import contextlib
import io
import sys
import time

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This notebook requires the retinanalysis Python 3.11 kernel; '
        f'got Python {sys.version.split()[0]} at {sys.executable}')
import_started = time.perf_counter()

import retinanalysis as ra
import numpy as np
import pandas as pd

from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import linear_equivalent_disc as led

print(f'Python {sys.version.split()[0]} | {sys.executable}')
print(f'Imports ready in {time.perf_counter() - import_started:.2f} s')

## 1. Find center-disc experiment dates and cells

Treat `LinearEquivalentDiscConeLin` and the newer, cone-linearizing
`LinearEquivalentDisc` as one center-disc analysis family. The compact table retains the exact
protocol name for provenance. For `LinearEquivalentDisc`, blocks without a saved
`linearizeCones` parameter belong to the older two-stimulus implementation and are excluded
before dates or cells are listed.


In [ ]:
CENTER_PROTOCOLS = (
    'LinearEquivalentDiscConeLin',
    'LinearEquivalentDisc',
)
PROTOCOL_LABEL = 'cone-linearized center disc'

protocol_cells = led.find_protocol_cells(CENTER_PROTOCOLS, show=False)
protocol_cells.insert(
    0, 'date_index', pd.factorize(protocol_cells['exp_name'], sort=False)[0] + 1)
print(f'{PROTOCOL_LABEL}: {len(protocol_cells)} protocol/cell rows across '
      f'{protocol_cells.exp_name.nunique()} experiments')
sc.scroll_table(protocol_cells, height=420)


### 1a. Choose a date and group its recordings

Choose one `date_index` from Section 1. Only that date is loaded in detail. Each group is one
cell × resolved recording mode × center-disc site × light setting; compatible blocks from
both protocol names can be pooled while the `protocols` column retains their provenance.
The metadata filter for old `LinearEquivalentDisc` blocks is applied again here.


In [ ]:
# Choose a date_index shown in Section 1.
DATE_INDEX = 1

date_rows = protocol_cells.loc[protocol_cells.date_index.eq(DATE_INDEX)]
if date_rows.empty:
    raise ValueError(f'date_index {DATE_INDEX} is not available for {PROTOCOL_LABEL}')
EXP_NAME = date_rows.exp_name.iloc[0]

# Only now load detailed blocks and resolve recording mode from the amplifier.
df_blocks = led.find_blocks(
    exp_names=[EXP_NAME], protocols=CENTER_PROTOCOLS, show=False)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    df_blocks = led.check_series_resistance(
        df_blocks, show=False, sample_series_resistance=True)
selected_blocks = df_blocks.copy()
if selected_blocks.empty:
    raise ValueError(f'{EXP_NAME!r} has no cone-linearized center-disc blocks')
groups = led.group_blocks(selected_blocks)


## 2. What the cell was shown

Continue from the date selected in Section 1a. The dropdown contains every recorded
`imageName` for that date; choosing one redraws an example center patch and its two equivalent
discs. The three stimuli use a common grey scale. The image is loaded from the van Hateren
resources in the turner package the same way `NaturalImageFlashProtocol.m` does.

A useful geometry check: the mean intensity inside the center aperture should match the
recorded `equivalentIntensity`.


In [ ]:
image_example = led.stimulus_example_widget(selected_blocks)
image_example

## 3. Analyze one cell condition

Enter the cell label, resolved `onlineAnalysis`, and numeric FilterWheel value. The first output
lists the matching block IDs and one row per `imageName`, including its epoch count and
`meanIntensity = maxIntensity × backgroundIntensity`.

Responses use the exact onset window from `preTime` through `preTime + stimTime`: spike count
for extracellular recordings and baseline-subtracted signed area (pA·s) for whole-cell
recordings. Each scatter point is one `(imageName, patchIndex)` pair; x and y error bars are
SEM across repeat epochs. The per-image and pooled figures show image vs standard disc in grey
and image vs cone-linearized disc in red, with a unity line. Patch indices that restart in a new
image remain separate. The final figure uses empirical NLI CDFs and a second panel with
the mean ± SEM across every image-specific patch for this cell. After Section 3a has saved this
exact block set once, rerunning after a kernel restart reloads those patch results instead of
re-reading traces and detecting spikes. Pass `reuse_saved=False` to `analyze_condition` when a
deliberate raw-data recomputation is needed.

In [ ]:
# Define one condition within the experiment selected in Section 1a.
CELL_LABEL = 'Cell1'
ONLINE_ANALYSIS = 'extracellular'  # 'extracellular', 'exc', or 'inh'
FILTER_WHEEL_VALUE = 0.0          # authoritative numeric FilterWheel value

condition_blocks = led.select_condition_blocks(
    selected_blocks, CELL_LABEL, ONLINE_ANALYSIS, FILTER_WHEEL_VALUE)
condition = led.analyze_condition(condition_blocks)
image_fig, pooled_fig, nli_fig = led.plot_condition(condition)

### 3a. Save this cell for population analysis

Save one compressed HDF5 record for the selected cell condition, similar to a MATLAB struct
containing arrays. Cell metadata is stored once, and the image summary and patch responses are
kept as typed arrays instead of tens of thousands of CSV rows. `condition_population_table()`
still expands the selected condition for display when needed. Each distinct date/cell/mode/FW
condition gets its own file; rerunning that exact condition replaces its file rather than
duplicating it. The file also becomes the fast restart source for Section 3 when the selected
block IDs match. Later,
`led.load_condition_outputs()` combines every saved cell into one population table.

In [ ]:
condition_output_path = led.save_condition_output(condition)
population_ready = led.condition_population_table(condition)
sc.scroll_table(
    population_ready, height=360,
    num_cols=('filter_wheel_ndf', 'patchIndex', 'maxIntensity',
              'meanIntensity', 'image_response', 'disc_response',
              'cone_disc_response', 'nli_image_vs_disc',
              'nli_image_vs_cone_disc'));

### 3b. Check saved cells

Read only the small metadata attributes from saved center-disc conditions—without loading the
patch arrays—and list date, cell label/type, resolved `onlineAnalysis`, and FilterWheel value.

In [ ]:
saved_cells = led.load_condition_index(protocol=CENTER_PROTOCOLS)
print(f'{len(saved_cells)} saved cell condition(s)')
sc.scroll_table(saved_cells, height=300, num_cols=('filter_wheel_ndf',));

## 5. Population by image and cell type

Load saved HDF5 conditions from either center-disc protocol name. Each row is one cell ×
FilterWheel × `imageName`: `meanIntensity` is x, and the two y values are mean patch NLI for
the ordinary and cone-linearized disc comparisons. Then group rows into `[500, 1500)`,
`[1500, 3500)`, `[3500, 6000)`, and `[6000, 20000]` within each cell type and plot population
mean NLI ± SEM. No averaging occurs across cells before this grouping.


In [ ]:
image_nli = led.load_condition_image_nli_summary(
    protocol=CENTER_PROTOCOLS)
print(f'{len(image_nli)} cell/image rows from {image_nli.cell_id.nunique()} cells')
image_nli_view = image_nli[[
    'exp_name', 'cell_label', 'cell_type', 'onlineAnalysis', 'protocol',
    'filter_wheel_ndf', 'imageName', 'meanIntensity', 'n_patches',
    'mean_nli_disc', 'mean_nli_cone_disc']]
sc.scroll_table(
    image_nli_view, height=360,
    num_cols=('filter_wheel_ndf', 'meanIntensity', 'n_patches',
              'mean_nli_disc', 'mean_nli_cone_disc'))
light_level_nli = led.summarize_image_nli_light_levels(image_nli)
print(f'{light_level_nli.n_cell_images.sum()} of {len(image_nli)} rows are within 500–20000')
sc.scroll_table(
    light_level_nli, height=300,
    num_cols=('light_min', 'light_max', 'meanIntensity', 'n_cells',
              'n_cell_images', 'mean_nli_disc', 'sem_nli_disc',
              'mean_nli_cone_disc', 'sem_nli_cone_disc'))
led.plot_image_nli_by_cell_type(
    image_nli, title_prefix='Cone-linearized center disc');


### Pooled patch NLI distributions

Pool every patch NLI from saved center-disc HDF5 conditions, without averaging by image or
cell. The left panel is a normalized 50-bin density and the right panel is the empirical CDF,
each comparing ordinary and cone-linearized discs.


In [ ]:
patch_nli = led.load_condition_patch_nli(
    protocol=CENTER_PROTOCOLS)
finite_patch_nli = np.isfinite(
    patch_nli[['nli_disc', 'nli_cone_disc']]).sum()
print(f"{finite_patch_nli['nli_disc']} standard and "
      f"{finite_patch_nli['nli_cone_disc']} cone-lin patch NLIs from "
      f'{len(patch_nli)} saved rows and {patch_nli.cell_id.nunique()} cells')
led.plot_pooled_patch_nli_distributions(
    patch_nli, bins=50, title_prefix='Cone-linearized center disc');


### Cell-level averages at ~1k and ~10k

For each cell, pool all patches across all `imageName`s within `[500, 1500)` (~1k) or
`[6000, 20000]` (~10k), then calculate one ordinary-disc and one cone-linearized mean NLI.
Faint points are those cell means; large points and error bars are population mean ± SEM
across cells, so a cell with more patches does not get extra population weight.


In [ ]:
cell_light_nli = led.summarize_cell_patch_nli_light_levels(patch_nli)
sc.scroll_table(
    cell_light_nli, height=320,
    num_cols=('light_min', 'light_max', 'meanIntensity', 'n_images',
              'n_patches', 'mean_nli_disc', 'mean_nli_cone_disc'))
led.plot_cell_patch_nli_by_light(
    cell_light_nli, title_prefix='Cone-linearized center disc');
